# 04 — Joins

**What's in this notebook:**
- The conceptual model of a join — pair-up-then-filter
- `INNER JOIN` — matches on both sides, plus `ON` vs `USING`
- Outer joins (`LEFT`, `RIGHT`, `FULL`) — preserve one or both sides, fill the rest with `NULL`
- `CROSS JOIN` and the accidental cartesian product
- `SELF JOIN` — when one side of the join is the same table
- Multi-table joins and how `NULL` behaves in join keys and outer-join results
- Common pitfalls to carry forward

This notebook uses `customers`, `orders`, and `prospects` from earlier notebooks, plus a small `employees` table we add for the self-join section.

In [ ]:
%load_ext sql
%sql duckdb:///learn.db

## 1. The conceptual model of a join

Conceptually, every join is the same two-step idea:

1. **Pair up every row of `A` with every row of `B`** — the cartesian product.
2. **Keep only the pairs that match the `ON` predicate.**

Real engines never actually materialize the full cartesian product — they use hash joins, sort-merge joins, or nested loops depending on the data — but the *result* is as-if they did. That's a useful mental model, because it explains why a missing `ON` clause produces all `N × M` rows and why a join key on a unique column produces at most `N` rows.

```
       customers              orders                 INNER JOIN
       +----+-------+         +----+----+--------+   c.id = o.cid
       | id | name  |         | id | cid| amount |   --------------
       +----+-------+         +----+----+--------+   Alice  29.99
       | 1  | Alice |  ──>    | 101| 1  | 29.99  |   Alice 149.50
       | 2  | Bob   |         | 102| 1  |149.50  |   Bob     9.99
       | 3  | Carol |         | 103| 2  |  9.99  |   (Carol drops —
       +----+-------+         +----+----+--------+   no matching order)
```

Outer joins change step 2 — instead of dropping unmatched rows, they keep them and fill the missing side with `NULL`.

## 2. INNER JOIN — and `ON` vs `USING`

`INNER JOIN` keeps only rows that have a match on both sides. The word `INNER` is optional; bare `JOIN` means the same thing — but writing `INNER JOIN` reads better in real code.

Two ways to specify the predicate:

- **`ON <boolean>`** — fully general. Any boolean expression works, including inequalities (`ON a.date BETWEEN b.start AND b.end`) and multi-column predicates.
- **`USING (col, ...)`** — shorthand when the join columns have the same name on both sides. Produces a single column in the output instead of two — `c.customer_id` and `o.customer_id` collapse into one `customer_id`.

Avoid `NATURAL JOIN` (auto-joins on every same-named column). It looks clever and breaks badly the moment somebody adds a new column with a colliding name.

In [ ]:
%%sql
-- INNER JOIN with ON: full control over the predicate.
SELECT c.name, o.order_id, o.amount
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.name, o.order_id;

In [ ]:
%%sql
-- Same join with USING. Note the result has one customer_id column, not two.
SELECT customer_id, name, order_id, amount
FROM customers
INNER JOIN orders USING (customer_id)
ORDER BY customer_id, order_id;

## 3. Outer joins — preserving one or both sides

Outer joins preserve unmatched rows instead of dropping them. There are three flavors:

- **`LEFT OUTER JOIN`** (commonly written as `LEFT JOIN`) — keeps every row from the left, fills the right with `NULL` when there is no match.
- **`RIGHT OUTER JOIN`** — mirror image. Keeps every row from the right. Rarely used in practice because you can always flip the table order and write a `LEFT JOIN` instead, which reads more naturally.
- **`FULL OUTER JOIN`** — keeps every row from both sides, filling either side with `NULL` where there is no match.

Use a `LEFT JOIN` when you want "every customer, with their orders if they have any" — and Carol, who has no orders, should still appear in the result with `NULL`s in the order columns.

In [ ]:
%%sql
-- LEFT JOIN: every customer, plus their orders (NULL when no order).
-- Carol shows up with NULL order fields.
SELECT c.name, o.order_id, o.amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.name, o.order_id;

In [ ]:
%%sql
-- FULL OUTER JOIN: customers and prospects matched by name.
-- Alice is on both sides; everyone else appears once with NULLs on the other side.
SELECT c.name AS customer_name,
       p.name AS prospect_name
FROM customers c
FULL OUTER JOIN prospects p ON c.name = p.name
ORDER BY COALESCE(c.name, p.name);

## 4. CROSS JOIN — and the accidental cartesian product

`CROSS JOIN` is the explicit cartesian product — every row on the left paired with every row on the right, no filter. It is occasionally what you want — generating a grid of `(customer, month)` pairs to left-join sales into, for instance.

The footgun is the **comma-separated `FROM` syntax** that predates explicit `JOIN` keywords: `FROM a, b` is a cross join. Add a `WHERE a.id = b.id` and it becomes an inner join, but forget the `WHERE` and you silently get `N × M` rows.

Modern explicit `JOIN ... ON` syntax makes this almost impossible — most engines reject `INNER JOIN` without `ON` as a syntax error. **Always use explicit join syntax in saved code.**

In [ ]:
%%sql
-- Explicit CROSS JOIN: 3 customers × 3 orders = 9 rows. Rarely what you want.
SELECT c.name, o.order_id
FROM customers c
CROSS JOIN orders o
ORDER BY c.name, o.order_id;

In [ ]:
%%sql
-- Accidental cartesian product via comma-FROM and a missing predicate.
-- Same 9 rows. This is how multi-million-row blowups happen in production.
SELECT c.name, o.order_id
FROM customers c, orders o
ORDER BY c.name, o.order_id;

## 5. SELF JOIN — when one side is the same table

A self join is just a regular join where both sides happen to be the same table. The mechanics are identical to any other join — the only requirement is that you **alias the table twice** so the engine can tell the two roles apart.

The classic use case is a self-referential hierarchy: an `employees` table where each row has a `manager_id` pointing at another row in the same table. To produce `(employee_name, manager_name)` pairs, you join `employees` to itself, treating one alias as the employee role and the other as the manager role.

In [ ]:
%%sql
-- A small org chart. Carol is the CEO (no manager).
DROP TABLE IF EXISTS employees;
CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    name        VARCHAR NOT NULL,
    manager_id  INTEGER REFERENCES employees(employee_id)
);
INSERT INTO employees VALUES
    (1, 'Carol', NULL),
    (2, 'Alice', 1),
    (3, 'Bob',   1),
    (4, 'Dave',  2);
SELECT * FROM employees;

In [ ]:
%%sql
-- Self join: pair each employee with their manager.
-- LEFT JOIN so Carol (no manager) still shows up.
SELECT e.name AS employee,
       m.name AS manager
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.employee_id
ORDER BY m.name NULLS FIRST, e.name;

## 6. Multi-table joins and `NULL` behavior

Joins chain — `A JOIN B ON ... JOIN C ON ...` is read left to right, with each `JOIN` adding another table to the running result. **The order you *write* the joins does not have to match the order the engine executes them** — the planner reorders based on cost. But for outer joins, the order is *semantically* meaningful, because `A LEFT JOIN B` and `B LEFT JOIN A` are different queries.

Two things about `NULL` in joins you should carry around with you:

- **`NULL` never matches in a join key.** `c.customer_id = o.customer_id` evaluates to `UNKNOWN` whenever either side is `NULL`, so the row is dropped (or, in an outer join, treated as unmatched). This is why join keys are almost always declared `NOT NULL`.
- **A `NULL` in an outer-join result is ambiguous.** Did the right side have no match, or did the right side have a match whose stored value happens to be `NULL`? You can't tell from the result alone. When this matters, either select a column declared `NOT NULL` from the right side (commonly the primary key) and check that for `NULL`, or pre-mark with a sentinel like `1 AS matched`.

In [ ]:
%%sql
-- A three-way pattern: customer, their orders, and whether they are also a prospect.
-- COALESCE lets us print a clean label instead of NULL.
SELECT c.name,
       COUNT(o.order_id)                AS orders,
       COALESCE(SUM(o.amount), 0)       AS total_spent,
       CASE WHEN p.prospect_id IS NULL
            THEN 'no'
            ELSE 'yes'
       END                              AS is_also_prospect
FROM customers c
LEFT JOIN orders    o ON o.customer_id = c.customer_id
LEFT JOIN prospects p ON p.name        = c.name
GROUP BY c.name, p.prospect_id
ORDER BY total_spent DESC;

## 7. Common pitfalls (carry forward)

- **Missing or wrong `ON` predicate** — the most common way to accidentally generate a cartesian product. Always check the row count of a join against your expectation.
- **Join keys containing `NULL`** — `NULL`s in a join key never match, so rows with `NULL` keys silently disappear from inner joins. Declare key columns `NOT NULL` whenever the data model allows it.
- **Outer-join `NULL`s are ambiguous** — a `NULL` in the right-side column can mean either "no match" or "matched, but the stored value is `NULL`". Disambiguate by checking a known-`NOT NULL` column from the right side (typically its primary key).
- **Filtering an outer-joined column in `WHERE` converts it back to an inner join.** A predicate like `WHERE o.amount > 10` after `LEFT JOIN orders o ON ...` drops the unmatched rows (because `NULL > 10` is `UNKNOWN`). If you want to keep them, move the predicate into the `ON` clause.
- **`SELECT *` across joins** — column-name collisions get auto-mangled by the engine in confusing ways. List the columns you want.
- **Comma-separated `FROM` without a `WHERE`** — instant cartesian product. Always use explicit `JOIN ... ON` in saved code.
- **Mixing aliases between joins** — once you alias a table, you must reference it by the alias everywhere in the query. `customers c JOIN orders o ON customers.customer_id = ...` is invalid in standard SQL — it must be `c.customer_id`.

Next up: **notebook 05 — Aggregation & Grouping**, where `GROUP BY`, `HAVING`, and the aggregate functions turn rows into summaries.